In [ ]:

!pip install -q pymupdf sentence-transformers faiss-cpu langchain-text-splitters google-genai gradio



In [ ]:

import fitz
import torch
import faiss
import gradio as gr
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google import genai

print("All libraries imported successfully!")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:

from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini API configured successfully!")


In [ ]:

from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("Uploaded file:", pdf_path)



In [ ]:

import fitz

# Open the uploaded PDF
document = fitz.open(pdf_path)

# Extract text from all pages
document_text = ""

for page_number, page in enumerate(document):
    text = page.get_text()
    document_text += text + "\n"

print("PDF processed successfully!")
print("Total pages:", len(document))
print("Total characters:", len(document_text))



In [ ]:

if len(document_text.strip()) > 0:
    print("SUCCESS: PDF text extraction completed.")
else:
    print("ERROR: No text found in PDF.")


In [ ]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Split document text
chunks = text_splitter.split_text(document_text)

print("Text chunking completed!")
print("Total chunks generated:", len(chunks))


In [ ]:

for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- CHUNK {i+1} ---\n")
    print(chunk)


In [ ]:

from sentence_transformers import SentenceTransformer

# Load embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")


In [ ]:

# Generate embeddings for all chunks
embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embeddings generated successfully!")
print("Embedding shape:", embeddings.shape)


In [ ]:

print("Total document chunks:", len(chunks))
print("Total embeddings:", len(embeddings))
print("Dimensions per embedding:", embeddings.shape[1])

if len(chunks) == len(embeddings):
    print("\nSUCCESS: Chunking and embedding completed!")
else:
    print("\nERROR: Chunk and embedding count mismatch.")



In [ ]:

import faiss
import numpy as np

print("FAISS imported successfully!")


In [ ]:

# Convert embeddings into float32
embedding_vectors = np.array(
    embeddings,
    dtype="float32"
)

# Normalize vectors
faiss.normalize_L2(embedding_vectors)

# Get embedding dimensions
dimension = embedding_vectors.shape[1]

# Create FAISS index
index = faiss.IndexFlatIP(dimension)

# Add embeddings to FAISS
index.add(embedding_vectors)

print("FAISS vector database created successfully!")
print("Total vectors stored:", index.ntotal)
print("Vector dimensions:", dimension)


In [ ]:

def retrieve_documents(query, top_k=3):

    # Convert user question into embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Normalize query vector
    faiss.normalize_L2(query_embedding)

    # Search FAISS index
    scores, indices = index.search(query_embedding, top_k)

    retrieved_results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1:
            continue

        retrieved_results.append({
            "chunk": chunks[idx],
            "similarity_score": float(score)
        })

    return retrieved_results


print("Retrieval function created successfully!")


In [ ]:

question = "What is Generative AI?"

results = retrieve_documents(question, top_k=3)

print("USER QUESTION:", question)
print("\nRETRIEVED DOCUMENT PASSAGES:\n")

for i, result in enumerate(results):

    print(f"--- RESULT {i+1} ---")
    print("Similarity Score:", round(result["similarity_score"], 4))
    print("Passage:", result["chunk"])
    print()


In [ ]:

test_questions = [
    "What is a Transformer architecture?",
    "Explain tokenization.",
    "What are embeddings?",
    "What is Retrieval-Augmented Generation?",
    "What is a hallucination in Generative AI?"
]

for q in test_questions:

    print("\nQUESTION:", q)

    retrieved = retrieve_documents(q, top_k=2)

    for item in retrieved:
        print("Similarity:", round(item["similarity_score"], 4))
        print("Passage:", item["chunk"][:250])


In [ ]:

print("========== FAISS DATABASE STATUS ==========")
print("Total document chunks:", len(chunks))
print("Total vectors stored:", index.ntotal)
print("Embedding dimensions:", dimension)

if index.ntotal == len(chunks):
    print("SUCCESS: FAISS database is ready!")
else:
    print("ERROR: Vector count mismatch!")


In [ ]:

from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully!")


In [ ]:

def generate_rag_answer(question):

    # Retrieve relevant passages
    retrieved_results = retrieve_documents(
        question,
        top_k=3
    )

    # Combine retrieved passages
    context = ""

    for i, result in enumerate(retrieved_results):
        context += f"\nPassage {i+1}:\n"
        context += result["chunk"] + "\n"

    # Create prompt
    prompt = f"""
You are a document-based question answering assistant.

Answer the user's question using ONLY the retrieved
document passages provided below.

Rules:
1. Do not add information from your general knowledge.
2. Do not invent facts, dates or statistics.
3. If the answer is not available in the passages,
   clearly say that the information is not available.
4. Give a clear and concise answer.

RETRIEVED CONTEXT:
{context}

USER QUESTION:
{question}

GENERATED ANSWER:
"""

    # Generate response using Gemini
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return {
        "question": question,
        "answer": response.text,
        "context": context,
        "retrieved_results": retrieved_results
    }


print("RAG answer generation function created!")


In [ ]:

question = "What is Generative AI?"

rag_result = generate_rag_answer(question)

print("USER QUESTION:")
print(rag_result["question"])

print("\nRETRIEVED CONTEXT:")
print(rag_result["context"])

print("\nGENERATED ANSWER:")
print(rag_result["answer"])


In [ ]:

def generate_rag_answer(question):

    # Step 1: Retrieve relevant passages
    retrieved_results = retrieve_documents(question, top_k=3)

    # Step 2: Prepare context
    context = ""

    for i, result in enumerate(retrieved_results):
        context += f"\nPassage {i+1}:\n{result['chunk']}\n"

    # Step 3: Prepare prompt
    prompt = f"""
You are a document-based question answering assistant.

Answer using ONLY the retrieved document passages.

Rules:
1. Do not use outside knowledge.
2. Do not invent facts, dates or statistics.
3. If information is unavailable, clearly state that.
4. Give a clear and concise answer.

RETRIEVED CONTEXT:
{context}

USER QUESTION:
{question}

GENERATED ANSWER:
"""

    # Step 4: Generate answer using updated Gemini model
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    # Step 5: Return complete results
    return {
        "question": question,
        "answer": response.text,
        "context": context,
        "retrieved_results": retrieved_results
    }


print("Updated RAG function successfully!")


In [ ]:

test_response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Explain Generative AI in two sentences."
)

print(test_response.text)


In [ ]:

!pip install -q groq



In [ ]:

from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq API configured successfully!")


In [ ]:

test_response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Explain Generative AI in two sentences."
        }
    ]
)

print(test_response.choices[0].message.content)


In [ ]:

def generate_rag_answer(question):

    # Retrieve relevant passages
    retrieved_results = retrieve_documents(question, top_k=3)

    # Prepare context
    context = ""

    for i, result in enumerate(retrieved_results):
        context += f"\nPassage {i+1}:\n{result['chunk']}\n"

    # Prepare prompt
    prompt = f"""
You are a document-based question answering assistant.

Answer using ONLY the retrieved passages.

1. Do not use outside knowledge.
2. Do not invent facts.
3. If information is unavailable, say so.
4. Give a clear answer.

RETRIEVED CONTEXT:
{context}

QUESTION:
{question}
"""

    # Generate response using Groq
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    answer = response.choices[0].message.content

    return {
        "question": question,
        "answer": answer,
        "context": context,
        "retrieved_results": retrieved_results
    }

print("RAG function updated successfully!")


In [ ]:

question = "What is Generative AI?"

rag_result = generate_rag_answer(question)

print("USER QUESTION:")
print(rag_result["question"])

print("\nRETRIEVED CONTEXT:")
print(rag_result["context"])

print("\nGENERATED ANSWER:")
print(rag_result["answer"])


In [ ]:

import json

def evaluate_rag_response(rag_result):

    question = rag_result["question"]
    answer = rag_result["answer"]
    context = rag_result["context"]

    prompt = f"""
You are an expert RAG response evaluator.

Your task is to evaluate the factual claims in a generated
answer against the retrieved document passages.

QUESTION:
{question}

RETRIEVED DOCUMENT PASSAGES:
{context}

GENERATED ANSWER:
{answer}

INSTRUCTIONS:

1. Break the generated answer into individual factual claims.
2. Evaluate every claim separately using ONLY the retrieved passages.
3. Assign exactly one label to each claim:

   Supported:
   The retrieved passages provide sufficient evidence.

   Unsupported:
   The retrieved passages do not provide enough evidence.

   Contradicted:
   The retrieved passages explicitly conflict with the claim.

4. Provide the exact supporting or conflicting evidence
   from the retrieved passages, if available.
5. Do not use your external knowledge.
6. Do not treat missing evidence as contradiction.
7. Do not omit factual claims.

Return ONLY valid JSON in this format:

{{
  "evaluations": [
    {{
      "claim": "Individual factual statement",
      "status": "Supported",
      "evidence": "Relevant passage from document",
      "reason": "Explanation for classification"
    }}
  ]
}}
"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    result = response.choices[0].message.content

    # Clean possible Markdown JSON formatting
    result = result.strip()

    if result.startswith("```"):
        result = result.split("\n", 1)[1]
        result = result.rsplit("```", 1)[0].strip()

    evaluation_data = json.loads(result)

    return evaluation_data


print("RAG evaluation function created successfully!")


In [ ]:

evaluation_result = evaluate_rag_response(rag_result)

print("========== CLAIM EVALUATION ==========\n")

for i, item in enumerate(evaluation_result["evaluations"]):

    print(f"CLAIM {i+1}:")
    print(item["claim"])

    print("\nSTATUS:")
    print(item["status"])

    print("\nEVIDENCE:")
    print(item["evidence"])

    print("\nREASON:")
    print(item["reason"])

    print("\n" + "-"*50)


In [ ]:

def calculate_support_score(evaluation_data):

    evaluations = evaluation_data["evaluations"]

    total_claims = len(evaluations)

    supported = sum(
        1 for item in evaluations
        if item["status"].lower() == "supported"
    )

    unsupported = sum(
        1 for item in evaluations
        if item["status"].lower() == "unsupported"
    )

    contradicted = sum(
        1 for item in evaluations
        if item["status"].lower() == "contradicted"
    )

    support_score = (
        (supported / total_claims) * 100
        if total_claims > 0 else 0
    )

    return {
        "total_claims": total_claims,
        "supported": supported,
        "unsupported": unsupported,
        "contradicted": contradicted,
        "support_score": round(support_score, 2)
    }


score_result = calculate_support_score(evaluation_result)

print("========== EVALUATION SUMMARY ==========")

print("Total Claims:", score_result["total_claims"])
print("Supported:", score_result["supported"])
print("Unsupported:", score_result["unsupported"])
print("Contradicted:", score_result["contradicted"])

print("Support Score:", score_result["support_score"], "%")


In [ ]:

# Create a controlled test answer

test_rag_result = {
    "question": "What is Generative AI?",

    "answer": """
    Generative AI is a category of artificial intelligence
    that creates new content based on learned patterns.

    Generative AI was invented in 1995 by Microsoft.
    """,

    "context": rag_result["context"]
}

test_evaluation = evaluate_rag_response(test_rag_result)

test_score = calculate_support_score(test_evaluation)

print("TEST EVALUATION:")
print(json.dumps(test_evaluation, indent=4))

print("\nTEST SUPPORT SCORE:")
print(test_score)


In [ ]:

!pip install -q gradio



In [ ]:

def run_rag_evaluation(question):

    if not question.strip():
        return (
            "Please enter a question.",
            "",
            "",
            "",
            ""
        )

    # Generate RAG answer
    rag_result = generate_rag_answer(question)

    answer = rag_result["answer"]

    # Evaluate generated answer
    evaluation = evaluate_rag_response(rag_result)

    # Calculate score
    score = calculate_support_score(evaluation)

    # Prepare retrieved passages
    retrieved_text = ""

    for i, item in enumerate(rag_result["retrieved_results"]):
        retrieved_text += (
            f"\n### Passage {i+1}\n"
            f"Similarity: {item['similarity_score']:.4f}\n\n"
            f"{item['chunk']}\n\n"
        )

    # Prepare claim-level results
    evaluation_text = ""

    for i, item in enumerate(evaluation["evaluations"]):

        evaluation_text += f"""
### Claim {i+1}: {item['status']}

**Statement:** {item['claim']}

**Evidence:** {item['evidence']}

**Reason:** {item['reason']}

---
"""

    # Prepare summary
    summary = f"""
### Evaluation Summary

- Total Claims: {score['total_claims']}
- Supported: {score['supported']}
- Unsupported: {score['unsupported']}
- Contradicted: {score['contradicted']}

# Support Score: {score['support_score']}%
"""

    return (
        answer,
        retrieved_text,
        evaluation_text,
        summary,
        "Evaluation completed successfully!"
    )


print("Application processing function created!")


In [ ]:

import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown(
        """
        # RAG Response Evaluation System

        ### Detect unsupported statements in AI-generated answers

        Upload your knowledge source in the notebook and
        evaluate answers against retrieved document passages.
        """
    )

    question_input = gr.Textbox(
        label="Enter your question",
        placeholder="What is Generative AI?",
        lines=2
    )

    evaluate_button = gr.Button(
        "Generate & Evaluate Answer",
        variant="primary"
    )

    status_output = gr.Markdown()

    with gr.Tab("Generated Answer"):
        answer_output = gr.Markdown()

    with gr.Tab("Retrieved Evidence"):
        retrieved_output = gr.Markdown()

    with gr.Tab("Claim Evaluation"):
        evaluation_output = gr.Markdown()

    with gr.Tab("Evaluation Summary"):
        summary_output = gr.Markdown()

    evaluate_button.click(
        fn=run_rag_evaluation,
        inputs=question_input,
        outputs=[
            answer_output,
            retrieved_output,
            evaluation_output,
            summary_output,
            status_output
        ]
    )

app.launch(share=True, debug=True)


/tmp/ipykernel_3307/2894153844.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ef2809aef3de2f03e1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
